###### Step 6: 光流与特征跟踪

这一节直接使用：

- `LK_optical_flow`, `HS_optical_flow`
- `flow_to_image`
- `Feature_tracker_sd`

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'tianmoucv').exists() and (candidate / 'tianmoucv_example').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError('找不到 tianmoucv 仓库根目录，请从仓库内打开本 Notebook。')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch


In [ ]:
from tianmoucv.data import TianmoucDataReader
from tianmoucv.isp import SD2XY, upsample_cross_conv
from tianmoucv.proc.opticalflow import HS_optical_flow, LK_optical_flow, flow_to_image
from tianmoucv.proc.tracking import Feature_tracker_sd
from tianmoucv.utils import ensure_tutorial_dataset

info = ensure_tutorial_dataset()
dataset = TianmoucDataReader(str(info.sample_dir), N=1, print_info=False)
if len(dataset) == 0:
    raise RuntimeError('官方测试样本中没有可读取的数据')
sample = dataset[min(5, len(dataset) - 1)]
rawdiff = sample['rawDiff'].float() 

## 1. 用一段 AOP 窗口估计经典光流

TD 保持原始时间差符号，并用交叉采样重建到 SD 网格。AOP 原始差分噪声和静态基线偏置较大，先做多帧平均、空间平滑，并以低梯度区域估计 TD 基线，再用时空信号强度抑制不可信的可视化区域；暗区不表示真实零运动。`ifInterploted=True` 输出与 RGB/COP 同尺寸，位移也换算为 COP 像素。

In [ ]:
window = rawdiff[:, :min(8, rawdiff.shape[1])]
grad_x, grad_y = SD2XY(window[1:].mean(dim=1))
temporal = upsample_cross_conv(window[0:1].mean(dim=1, keepdim=True))[0, 0]
derivatives = np.stack([grad_x.numpy(), grad_y.numpy(), temporal.numpy()])
derivatives = np.stack([cv2.GaussianBlur(channel, (0, 0), 1.4) for channel in derivatives]).astype(np.float32)
spatial_activity = cv2.GaussianBlur(np.linalg.norm(derivatives[:2], axis=0), (0, 0), 4.0)
flat = spatial_activity <= np.median(spatial_activity)
derivatives[2] -= float(np.median(derivatives[2][flat]))
sd_acc = torch.from_numpy(derivatives[:2])
td_acc = torch.from_numpy(derivatives[2:3])
temporal_activity = cv2.GaussianBlur(np.abs(derivatives[2]), (0, 0), 4.0)
temporal_low = max(0.25, 1.75 * float(np.quantile(temporal_activity, 0.25)))
spatial_low = max(0.5, 1.5 * float(np.quantile(spatial_activity, 0.25)))
confidence = np.minimum(np.clip(temporal_activity / temporal_low - 1, 0, 1),
                        np.clip(spatial_activity / spatial_low - 1, 0, 1))
confidence = cv2.resize(confidence, (sd_acc.shape[-1] * 2, sd_acc.shape[-2] * 2))
confidence = cv2.GaussianBlur(confidence, (0, 0), 2.0)

flow_lk = LK_optical_flow(sd_acc, td_acc, win=31, stride=3, ifInterploted=True)
flow_hs = HS_optical_flow(sd_acc, td_acc, ifInterploted=True, maxIteration=50, scales=3, labmda=14)

flow_lk_rgb = cv2.cvtColor(flow_to_image(flow_lk.permute(1, 2, 0).numpy()), cv2.COLOR_BGR2RGB)
flow_hs_rgb = cv2.cvtColor(flow_to_image(flow_hs.permute(1, 2, 0).numpy()), cv2.COLOR_BGR2RGB)
flow_lk_rgb = (flow_lk_rgb * confidence[..., None]).astype(np.uint8)
flow_hs_rgb = (flow_hs_rgb * confidence[..., None]).astype(np.uint8)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.imshow(sample['F0'].numpy())
plt.title('F0')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(flow_lk_rgb)
plt.title('LK_optical_flow')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(flow_hs_rgb)
plt.title('HS_optical_flow')
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. 基于 SD 的特征跟踪

In [ ]:
tracker = Feature_tracker_sd(max_features=150, max_history=100,
                             max_trail_age=100, redetect_interval=5)
tracked_frames = 0
track_vis = None

for sample_index in range(min(2, len(dataset))):
    sample_key, relative_index = dataset.locateSample(sample_index)
    rawdiff = dataset.packRead(relative_index, sample_key, needPreProcess=False)['rawDiff']
    for frame_index in range(rawdiff.shape[1]):
        grad_x, grad_y = SD2XY(rawdiff[1:, frame_index].permute(1, 2, 0))
        magnitude = torch.hypot(grad_x, grad_y)
        _, track_vis = tracker.update(magnitude, viz=True)
        tracked_frames += 1

plt.figure(figsize=(8, 4))
plt.imshow(cv2.cvtColor(track_vis, cv2.COLOR_BGR2RGB))
plt.title(f'Feature_tracker_sd after {tracked_frames} AOP frames')
plt.axis('off')
plt.show()